<a href="https://colab.research.google.com/github/soukeyna898/Pret_immobilier/blob/main/Perso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
=====================================================================
 MODELE DE PREDICTION D'APPROBATION DE PRET IMMOBILIER
=====================================================================
Objectif : predire si une demande de pret immobilier sera APPROUVEE
ou REFUSEE, a partir des caracteristiques du demandeur.

Ce script :
  1. Genere un jeu de donnees synthetique realiste
  2. Pretraite les donnees (encodage, mise a l'echelle)
  3. Entraine deux modeles (Regression Logistique + Random Forest)
  4. Evalue et compare les performances
  5. Sauvegarde le meilleur modele
  6. Fournit une fonction pour predire un nouveau dossier

Pour l'utiliser avec VOS propres donnees : remplacez la fonction
generate_synthetic_data() par un pd.read_csv("votre_fichier.csv")
en gardant les memes noms de colonnes (ou adaptez FEATURES ci-dessous).
=====================================================================
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ---------------------------------------------------------------------
# 1. GENERATION DES DONNEES SYNTHETIQUES
# ---------------------------------------------------------------------
def generate_synthetic_data(n_samples: int = 3000) -> pd.DataFrame:
    """
    Cree un jeu de donnees synthetique mais realiste pour l'octroi
    de prets immobiliers. La probabilite d'approbation depend
    logiquement des variables (score de credit, ratio d'endettement,
    apport personnel, etc.) pour que le modele ait quelque chose
    de coherent a apprendre.
    """
    revenu_annuel = np.random.normal(65000, 25000, n_samples).clip(15000, 250000)
    age = np.random.randint(21, 70, n_samples)
    anciennete_emploi = np.random.exponential(6, n_samples).clip(0, 40)
    score_credit = np.random.normal(680, 80, n_samples).clip(300, 850)

    montant_pret = np.random.normal(220000, 90000, n_samples).clip(30000, 800000)
    valeur_bien = montant_pret * np.random.uniform(1.05, 1.4, n_samples)
    apport_personnel_pct = np.random.uniform(0, 0.4, n_samples)

    dettes_mensuelles = np.random.normal(600, 400, n_samples).clip(0, 5000)
    revenu_mensuel = revenu_annuel / 12
    ratio_endettement = (dettes_mensuelles + montant_pret * 0.005) / revenu_mensuel

    nb_personnes_a_charge = np.random.randint(0, 5, n_samples)
    statut_marital = np.random.choice(["celibataire", "marie", "divorce"], n_samples,
                                       p=[0.4, 0.45, 0.15])
    niveau_education = np.random.choice(["secondaire", "college", "universite"], n_samples,
                                         p=[0.25, 0.35, 0.40])
    type_emploi = np.random.choice(["salarie", "independant", "fonctionnaire"], n_samples,
                                    p=[0.55, 0.25, 0.20])

    df = pd.DataFrame({
        "revenu_annuel": revenu_annuel.round(0),
        "age": age,
        "anciennete_emploi": anciennete_emploi.round(1),
        "score_credit": score_credit.round(0),
        "montant_pret": montant_pret.round(0),
        "valeur_bien": valeur_bien.round(0),
        "apport_personnel_pct": (apport_personnel_pct * 100).round(1),
        "ratio_endettement": ratio_endettement.round(3),
        "nb_personnes_a_charge": nb_personnes_a_charge,
        "statut_marital": statut_marital,
        "niveau_education": niveau_education,
        "type_emploi": type_emploi,
    })

    # ---- Construction d'une probabilite d'approbation "logique" ----
    score = (
        0.004 * (df["score_credit"] - 600)
        - 3.0 * df["ratio_endettement"]
        + 0.05 * df["apport_personnel_pct"]
        + 0.03 * df["anciennete_emploi"]
        - 0.000003 * df["montant_pret"]
        + 0.1 * (df["type_emploi"] == "fonctionnaire").astype(int)
        - 0.15 * (df["type_emploi"] == "independant").astype(int)
    )
    proba_approbation = 1 / (1 + np.exp(-score))  # fonction sigmoide
    df["pret_approuve"] = (np.random.rand(n_samples) < proba_approbation).astype(int)

    return df


# ---------------------------------------------------------------------
# 2. PRETRAITEMENT
# ---------------------------------------------------------------------
CATEGORICAL_FEATURES = ["statut_marital", "niveau_education", "type_emploi"]
NUMERIC_FEATURES = [
    "revenu_annuel", "age", "anciennete_emploi", "score_credit",
    "montant_pret", "valeur_bien", "apport_personnel_pct",
    "ratio_endettement", "nb_personnes_a_charge",
]
TARGET = "pret_approuve"


def preprocess(df: pd.DataFrame, encoders: dict = None, scaler: StandardScaler = None, fit: bool = True):
    """Encode les variables categorielles et met a l'echelle les variables numeriques."""
    df = df.copy()
    encoders = encoders or {}

    for col in CATEGORICAL_FEATURES:
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            encoders[col] = le
        else:
            le = encoders[col]
            df[col] = le.transform(df[col])

    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

    if fit:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    return X_scaled, encoders, scaler


# ---------------------------------------------------------------------
# 3. ENTRAINEMENT ET EVALUATION
# ---------------------------------------------------------------------
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n--- {name} ---")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision : {precision_score(y_test, y_pred):.3f}")
    print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
    print(f"F1-score  : {f1_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.3f}")
    print("Matrice de confusion :")
    print(confusion_matrix(y_test, y_pred))
    print("\nRapport detaille :")
    print(classification_report(y_test, y_pred, target_names=["Refuse", "Approuve"]))

    return roc_auc_score(y_test, y_proba)


def main():
    print("Generation des donnees synthetiques...")
    df = generate_synthetic_data(3000)
    print(f"Jeu de donnees : {df.shape[0]} demandes, taux d'approbation = {df[TARGET].mean():.1%}")

    X, encoders, scaler = preprocess(df, fit=True)
    y = df[TARGET].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # --- Modele 1 : Regression logistique (simple, interpretable) ---
    log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    log_reg.fit(X_train, y_train)
    auc_lr = evaluate_model("Regression Logistique", log_reg, X_test, y_test)

    # --- Modele 2 : Random Forest (plus performant en general) ---
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    auc_rf = evaluate_model("Random Forest", rf, X_test, y_test)

    # --- Selection du meilleur modele ---
    best_model, best_name = (rf, "Random Forest") if auc_rf >= auc_lr else (log_reg, "Regression Logistique")
    print(f"\n>>> Meilleur modele retenu : {best_name} (AUC = {max(auc_rf, auc_lr):.3f})")

    # --- Importance des variables (Random Forest) ---
    importances = pd.Series(rf.feature_importances_, index=NUMERIC_FEATURES + CATEGORICAL_FEATURES)
    print("\nImportance des variables (Random Forest) :")
    print(importances.sort_values(ascending=False).round(3))

    # --- Sauvegarde du modele et des objets de pretraitement ---
    joblib.dump(best_model, "modele_pret.pkl")
    joblib.dump(encoders, "encoders.pkl")
    joblib.dump(scaler, "scaler.pkl")
    print("\nModele sauvegarde : modele_pret.pkl, encoders.pkl, scaler.pkl")

    return best_model, encoders, scaler


# ---------------------------------------------------------------------
# 4. PREDICTION SUR UN NOUVEAU DOSSIER
# ---------------------------------------------------------------------
def predict_new_application(model, encoders, scaler, demande: dict):
    """
    Predit l'approbation d'un nouveau dossier de pret.

    Exemple de `demande` :
    {
        "revenu_annuel": 72000,
        "age": 34,
        "anciennete_emploi": 5,
        "score_credit": 710,
        "montant_pret": 250000,
        "valeur_bien": 310000,
        "apport_personnel_pct": 15,
        "ratio_endettement": 0.28,
        "nb_personnes_a_charge": 1,
        "statut_marital": "marie",
        "niveau_education": "universite",
        "type_emploi": "salarie",
    }
    """
    df_new = pd.DataFrame([demande])
    X_new, _, _ = preprocess(df_new, encoders=encoders, scaler=scaler, fit=False)

    prediction = model.predict(X_new)[0]
    proba = model.predict_proba(X_new)[0, 1]

    resultat = "APPROUVE" if prediction == 1 else "REFUSE"
    print(f"\nResultat : {resultat} (probabilite d'approbation = {proba:.1%})")
    return resultat, proba


if __name__ == "__main__":
    model, encoders, scaler = main()

    # --- Exemple d'utilisation sur un nouveau dossier ---
    print("\n" + "=" * 60)
    print("EXEMPLE : prediction sur un nouveau dossier")
    print("=" * 60)
    exemple_demande = {
        "revenu_annuel": 72000,
        "age": 34,
        "anciennete_emploi": 5,
        "score_credit": 710,
        "montant_pret": 250000,
        "valeur_bien": 310000,
        "apport_personnel_pct": 15,
        "ratio_endettement": 0.28,
        "nb_personnes_a_charge": 1,
        "statut_marital": "marie",
        "niveau_education": "universite",
        "type_emploi": "salarie",
    }
    predict_new_application(model, encoders, scaler, exemple_demande)

Generation des donnees synthetiques...
Jeu de donnees : 3000 demandes, taux d'approbation = 44.6%

--- Regression Logistique ---
Accuracy  : 0.682
Precision : 0.656
Recall    : 0.599
F1-score  : 0.626
ROC-AUC   : 0.747
Matrice de confusion :
[[249  84]
 [107 160]]

Rapport detaille :
              precision    recall  f1-score   support

      Refuse       0.70      0.75      0.72       333
    Approuve       0.66      0.60      0.63       267

    accuracy                           0.68       600
   macro avg       0.68      0.67      0.67       600
weighted avg       0.68      0.68      0.68       600


--- Random Forest ---
Accuracy  : 0.693
Precision : 0.663
Recall    : 0.633
F1-score  : 0.648
ROC-AUC   : 0.740
Matrice de confusion :
[[247  86]
 [ 98 169]]

Rapport detaille :
              precision    recall  f1-score   support

      Refuse       0.72      0.74      0.73       333
    Approuve       0.66      0.63      0.65       267

    accuracy                           0.69  